# Orbit Wars Kaggle notebook - tutorial + submission builder

This notebook is meant to do two jobs at once:

1. **Teach the game in plain English** so you can watch a replay and understand what is happening.
2. **Produce a valid Kaggle submission archive** from the same `orbit_wars_agent.py` file that we test locally.

The companion strategy file `orbit_wars_agent.py` should sit in the same folder as this notebook.

## Orbit Wars in plain English

Think of Orbit Wars as a tiny real-time strategy game on a **100x100 board**.

- The **sun** sits in the center at `(50, 50)` and destroys fleets that cross it.
- **Planets** are resource nodes. If you own a planet, it creates new ships every turn.
- Some planets are **static** and some **orbit the sun**, so the target you aim at might move before your fleet arrives.
- A move is `[from_planet_id, angle_in_radians, num_ships]`.
- Bigger fleets move faster, but they are still destroyed if they hit the sun or fly off the board.
- When fleets arrive together, combat is resolved by **largest stack vs second-largest stack**.

### What usually wins games

A good bot tries to:

- expand to profitable neutral planets,
- not strip too many ships from planets that need defense,
- send fleets on **sun-safe** routes,
- time attacks so the target has not already produced too many extra ships by arrival.

The stronger public baselines go even further with **multi-source attacks, proactive defense, and more detailed route planning**. This notebook keeps the strategy readable, but still includes orbit prediction, intercept solving, surplus calculation, comet racing, and bounded multi-launch turns.

In [ ]:
%pip install -q kaggle kaggle-environments

In [ ]:
import os
from pathlib import Path

from kaggle.api.kaggle_api_extended import KaggleApi

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_key = kaggle_dir / "kaggle.json"
assert kaggle_key.exists(), f"Missing credentials file: {kaggle_key}"
os.chmod(kaggle_key, 0o600)

api = KaggleApi()
api.authenticate()
print("Kaggle authentication OK")
print(f"Using credentials from {kaggle_key}")

In [ ]:
import importlib
import math
import tarfile
from pathlib import Path

from kaggle_environments import make

AGENT_SOURCE_PATH = Path("orbit_wars_agent.py")
assert AGENT_SOURCE_PATH.exists(), f"Missing agent source file: {AGENT_SOURCE_PATH.resolve()}"

import orbit_wars_agent as strategy

strategy = importlib.reload(strategy)
AGENT_SOURCE = AGENT_SOURCE_PATH.read_text(encoding="utf-8")
agent = strategy.agent
DecisionLogic = strategy.DecisionLogic
PositionPredictor = strategy.PositionPredictor
InterceptSolver = strategy.InterceptSolver
ForwardSimulator = strategy.ForwardSimulator
SurplusCalculator = strategy.SurplusCalculator
ScoringFunction = strategy.ScoringFunction
Planet = strategy.Planet
Fleet = strategy.Fleet
CENTER = strategy.CENTER
ROTATION_RADIUS_LIMIT = strategy.ROTATION_RADIUS_LIMIT
get_field = strategy.get_field

def final_ship_scores(env):
    obs = env.state[0].observation
    scores = [0 for _ in env.state]
    for planet in get_field(obs, "planets", []):
        owner = int(planet[1])
        if owner != -1:
            scores[owner] += int(planet[5])
    for fleet in get_field(obs, "fleets", []):
        scores[int(fleet[1])] += int(fleet[6])
    return scores

def describe_observation(obs):
    planets = [Planet(*planet) for planet in get_field(obs, "planets", [])]
    player = int(get_field(obs, "player", 0))
    my_planets = [planet for planet in planets if planet.owner == player]
    enemy_planets = [planet for planet in planets if planet.owner not in (-1, player)]
    neutral_planets = [planet for planet in planets if planet.owner == -1]
    orbiting_planets = []
    static_planets = []
    for planet in planets:
        orbital_radius = math.hypot(planet.x - CENTER, planet.y - CENTER)
        if orbital_radius + planet.radius < ROTATION_RADIUS_LIMIT:
            orbiting_planets.append(planet)
        else:
            static_planets.append(planet)
    print(f"Step: {get_field(obs, 'step', 0)}")
    print(f"Your planets: {len(my_planets)} | Enemy planets: {len(enemy_planets)} | Neutral planets: {len(neutral_planets)}")
    print(f"Orbiting planets: {len(orbiting_planets)} | Static planets: {len(static_planets)}")
    print(f"Angular velocity: {get_field(obs, 'angular_velocity', 0.0):.4f} radians/turn")

print(f"Loaded agent from {AGENT_SOURCE_PATH.resolve()}")

## What this bot is trying to do

The agent follows a priority stack each turn:

1. **Defend** planets that would otherwise fall.
2. **Snipe** recently captured neutral planets while they are still weak.
3. **Grab comets** when we can beat the enemy there.
4. **Expand** to strong neutral planets.
5. **Attack** enemy planets when we have enough spare ships.
6. **Reinforce** the part of our empire that is under the most pressure.

The important upgrade in this version is that it can now launch **multiple fleets in one turn**, but it does so in a **bounded** way so it stays within the Kaggle time limit and does not empty every planet at once.

In [ ]:
# Tutorial preview: look at a fresh board before any fleets launch.
# The first blank step initializes the random map.
preview_env = make("orbit_wars", debug=True)
preview_env.reset(2)
preview_env.step([[], []])
preview_obs = preview_env.state[0].observation
describe_observation(preview_obs)

opening_moves = agent(preview_obs, preview_env.configuration)
print("\nBot opening moves:", opening_moves)
if opening_moves:
    print("The bot already sees one or more early launches it likes.")
else:
    print("No turn-0 launch is also normal: sometimes waiting one turn gives a cleaner intercept.")

print("\nWhat to look for in the board preview:")
print("- Planets near the center usually orbit.")
print("- Straight lines through the center are dangerous because fleets die in the sun.")
print("- High-production neutral planets are valuable expansion targets.")
preview_env.render(mode="ipython", width=800, height=600)

In [ ]:
# Cell 1 - Sanity checks
orbiting_target = Planet(1, -1, 60.0, 60.0, 2.0, 8, 2)
source_planet = Planet(0, 0, 80.0, 80.0, 3.0, 40, 3)
initial_planets = [source_planet, orbiting_target]
predictor = PositionPredictor(0, 0.05, initial_planets, initial_planets, [])
solver = InterceptSolver(predictor)
solution = solver.solve_intercept(
    source_planet.x,
    source_planet.y,
    400,
    orbiting_target,
    0,
    0.05,
    initial_planets,
)
assert solution is not None
predicted_target = predictor.predict_planet_pos(orbiting_target, solution[1])
expected_angle = math.atan2(
    predicted_target[1] - source_planet.y,
    predicted_target[0] - source_planet.x,
)
angle_delta = ((solution[0] - expected_angle + math.pi) % (2 * math.pi)) - math.pi
assert abs(angle_delta) < 1e-6
assert solver.check_sun_collision(80.0, 50.0, math.pi, 2.0, 20) is True

threatened_planet = Planet(2, 0, 80.0, 50.0, 3.0, 5, 1)
threat_predictor = PositionPredictor(0, 0.0, [threatened_planet], [threatened_planet], [])
enemy_fleet = Fleet(0, 1, 76.0, 50.0, 0.0, 99, 20)
surplus = SurplusCalculator(0).get_surplus(
    threatened_planet,
    [],
    [enemy_fleet],
    threat_predictor,
    turns_ahead=10,
)
assert surplus == 0
print("Sanity checks passed.")
print("Intercept solution:", solution)
print("Threatened surplus:", surplus)

## How to read the replay

When you run the next cell, watch for these ideas:

- **Expansion:** early fleets should usually head toward useful neutral planets.
- **Sun safety:** fleets should avoid lines that cut across the center.
- **Reinforcement:** back planets may send ships toward the contested frontier.
- **Multiple launches:** one turn can now contain several fleets from different planets.

If the replay feels chaotic, focus on just one question at a time: **which planets are producing ships for us, and which fleets are trying to change that?**

In [ ]:
# Cell 2 - Run a local game, print the result, and render the replay.
env = make("orbit_wars", debug=True)
env.run([agent, "random"])

final = env.steps[-1]
for i, state in enumerate(final):
    print(f"Player {i}: reward={state.reward}, status={state.status}")
print("Final ship scores:", final_ship_scores(env))

assert final[0].reward > 0

print("\nReplay tip: look for the turns where our bot launches from more than one planet.")
env.render(mode="ipython", width=800, height=600)

In [ ]:
# Cell 3 - Run 5 games vs random, print win rate
wins = 0
results = []
for game_idx in range(5):
    env = make("orbit_wars", debug=True)
    env.run([agent, "random"])
    rewards = [state.reward for state in env.steps[-1]]
    results.append(rewards)
    if rewards[0] > rewards[1]:
        wins += 1
win_rate = wins / 5
print("Results:", results)
print("Win rate:", win_rate)

In [ ]:
# Cell 4 - Self-play game (agent vs agent), verify no crashes
env = make("orbit_wars", debug=True)
env.run([agent, agent])
self_play_rewards = [state.reward for state in env.steps[-1]]
print("Self-play rewards:", self_play_rewards)

In [ ]:
# Submission cell (last cell)
submission_dir = Path("submission")
submission_dir.mkdir(exist_ok=True)
(submission_dir / "main.py").write_text(AGENT_SOURCE, encoding="utf-8")

archive_path = Path("submission.tar.gz")
with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(submission_dir / "main.py", arcname="main.py")

print("Ready to submit!")
# !kaggle competitions submit -c orbit-wars -f submission.tar.gz -m "algo bot v1"